# Workflow 2: Data exploration: Spectra visualization, PCA

## Inputs

In [ ]:
#Import libraries
import os
import pandas as pd
import matplotlib.pyplot as plt
import exploration_modelling_functions as ef

In [ ]:
# === Initialize Project Paths and Folders ===

# Define the project name and base working directory
project_name="check_2908"
working_directory="C:/Users/Pheno/Documents/database_almondcv3/"

# Define the project and results directory

project_directory=os.path.join(working_directory, project_name)
results_directory=os.path.join(project_directory, f"Results_exploration_{project_name}")

# Create the necessary directories if they don't already exist
if not os.path.exists(project_directory):
    
    os.makedirs(project_directory)
if not os.path.exists(results_directory):

    os.makedirs(results_directory)

# Define the spectral data path
file_data_path= r"C:\Users\Pheno\Documents\database_almondcv3\RESULTADOS_PAPER\Datasets_curated\Kernel\all_results_joined.txt"

# Define the features/labels data path
file_labels_path=os.path.join(working_directory, r"Training_dataset_paper_hyper\LABELS_familia.txt")



## Data integration

In [ ]:
# === Integrates spectral data and features/labels for data exploration ===

# Import spectral data (tab-delimited file, force column "ID" to be a categorical type)
df_all_results = pd.read_csv(file_data_path, delimiter='\t', dtype={'ID': 'category'})

# Extract all preprocessing methods (columns that start with "Mean" or "Median")
preprocessing_methods = df_all_results.filter(regex="^(Mean|Median)").columns.tolist()

# Import features/labels (tab-delimited file, "ID" also as categorical to ensure consistent merging)
df_labels = pd.read_csv(file_labels_path, delimiter='\t', dtype={'ID': 'category'})

# Merge spectral data with labels/features on "ID"
# - on='ID' → join by sample identifier
# - how='left' → keep all rows from df_all_results, add matching labels from df_labels
df_all_results = pd.merge(df_all_results, df_labels, on='ID', how='left')

# Display the merged DataFrame (uncomment the line below if you want to preview the data in the console)
# df_all_results

# Display the list of preprocessing methods (uncomment the line below to preview it)
# preprocessing_methods

## Data curation 

In [ ]:
# Remove samples
# Create an empty list of IDs to remove.
remove_ids = []
# Keep only the rows where the "ID" column is NOT in remove_ids
df_all_results = df_all_results[~df_all_results["ID"].isin(remove_ids)]


In [ ]:
# Remove duplicates

# Count how many times each combination of "Band" and "ID" appears
duplicates = df_all_results.groupby(["Band", "ID"]).size().reset_index(name='count')

# Print only the rows where the same ("Band", "ID") pair appears more than once
print(duplicates[duplicates["count"] > 1])

# Remove duplicate rows, keeping only the first occurrence of each ("Band", "ID") pair
df_all_results = df_all_results.drop_duplicates(subset=["Band", "ID"])


### COW - Correlation Optimized Warping

In [ ]:
"""
Applies COW (Correlation Optimized Warping) to multiple spectral metrics in a DataFrame, overwriting the original columns.

Parameters:
- df_all_results: DataFrame containing 'Band', sample_id_col, and spectral metrics.
- preprocessing_metrics: list of columns to which COW will be applied.
- sample_id_col: name of the column identifying the samples (default: 'ID').
- n_segments: number of segments the spectrum is divided into; more segments = finer alignment but higher risk of overfitting
- slack: maximum allowed stretch/compression per segment; higher slack = more flexibility but can distort the signal
- reference: 'mean', 'median', or 'sample:<ID>' to use a specific sample as reference.

Returns:
- DataFrame with the specified metrics aligned (overwrites the original columns).
"""
df_all_results = ef.apply_cow_to_multiple_metrics(df_all_results=df_all_results, preprocessing_metrics=preprocessing_methods,
                                            sample_id_col='ID', n_segments=10, slack=1, reference="sample:E_73_A")

In [ ]:
# Export df aligned

# df_all_results.to_csv(f"{os.path.dirname(file_data_path)}/all_results_joined_alignedcowto_E_73_A.txt", index=False, sep="\t")


## Spectrum visualization

In [ ]:
unique_ids=df_all_results['ID'].unique()

In [ ]:
# === Visualize and save preprocessed spectral data ===
# This block generates plots of the selected preprocessing metrics for all or individual samples.
# It can color the spectra according to a specific feature (e.g., Fats content),
# optionally mean-center the spectra, set axis limits, and mark specific bands with a vertical line.
# The resulting plots are saved to the specified results directory.
"""
Generates and saves plots for a single selected preprocessing metric.

:param df_all_results: DataFrame containing the metric data.
:param unique_ids: List of unique IDs to process.
:param preprocessing_metric: Name of the preprocessing metric to plot (e.g., 'Mean_SpectralValue').
:param results_directory: Folder where the generated images will be saved.
:param ids: Name of the column in the DataFrame containing the sample IDs.
:param plot_type: Type of plot to generate ("individual", "combined", "both").
:param color_col: Name of the column used to define line colors. Can be quantitative or categorical.
:param xlim: Limits of the X axis (default between 900 and 1748).
:param ylim: Limits of the Y axis. If not provided, it will be auto-scaled.
:param vline: X-axis value where a vertical line will be drawn.
:param mean_centering: If True, the mean per band and preprocessing metric is subtracted before plotting.
"""

ef.plot_preprocessing_results(df_all_results, unique_ids, preprocessing=preprocessing_methods, results_directory=results_directory,
                               ids="ID", plot_type="combined", color_col="Fats", mean_centering=True,  xlim=None, ylim=None, vline=1600)

In [ ]:
# Plots the mean spectrum and standard deviation for all samples.
ef.plot_mean_std_spectrum(df_all_results, preprocessing_metric="Mean_SNV", results_directory=results_directory)

## PCA

In [ ]:
# 1️⃣ Columns to keep fixed (metadata and traits of interest for PCA or analysis)
id_vars_list = [
    'Session', 'Picture_name', 'ID', 'Band', 'Name', 'Farm', 
    'Fats', 'Moisture', 'Fiber', 'Protein', 'Sucrose', 
    'C_16', 'C_18', 'C_18_1', 'C_18_2', 'Cholesterol',
    'Campesterol', 'Stigmasterol', 'Betasitosterol', 
    'Delta7_Stigmasterol', 'Total_Sterols'
]

# 2️⃣ Spectral columns (preprocessed values we want to analyze/flatten)
spectral_cols = df_all_results.filter(regex="^(Mean|Median)").columns.tolist()

# 3️⃣ Convert the DataFrame from wide to long format
# - id_vars: columns to keep fixed (metadata, traits of interest for PCA)
# - value_vars: spectral metrics to flatten (Mean/Median columns)
# - var_name: new column with the name of the metric
# - value_name: new column with the corresponding value
df_long = pd.melt(
    df_all_results,
    id_vars=id_vars_list,      # fixed columns
    value_vars=spectral_cols,  # columns to flatten
    var_name='Metric',         # new column with metric names
    value_name='Value'         # new column with metric values
)

# Uncomment to display
# df_long

In [ ]:
# === Remove outlier samples from the DataFrame ===

# List of outlier IDs to exclude
outliers = [
    "ERROR",  # Example of an ID with an error in the data
    "na",     # Example of a missing or invalid ID
    "a"       # Example of another invalid or unexpected ID
]

# Filter the DataFrame to remove any rows where the 'ID' column matches an outlier
# - ~df_long["ID"].isin(outliers) creates a boolean mask where True means the ID is NOT in the outliers list

df_long = df_long[~df_long["ID"].isin(outliers)].reset_index(drop=True)


In [ ]:
"""
=== PCA Preprocessing and Visualization Function ===

This function performs Principal Component Analysis (PCA) on a selected spectral metric 
from a long-format DataFrame and provides multiple visualization and outlier detection options. 

Key functionalities:
- Reduces the dimensionality of spectral data (samples x bands).
- Calculates Q residuals for each sample to detect deviations from the PCA model.
- Optionally calculates Hotelling's T² for multivariate outlier detection.
- Visualizes the top influential variables in the component space (based on loadings).
- Generates PCA scatter plots colored by a metadata variable.
- Plots cumulative explained variance for the computed components.

Arguments:
- df_long: long-format DataFrame with columns ['Metric', 'Value', sample metadata...].
- preprocessing_metric: spectral metric/column to use for PCA (e.g., 'Mean_SG1_W15_P2').
- unique_sample_variable: column identifying individual samples (e.g., 'ID').
- color_variable: optional; column used to color samples in the PCA plot (numeric or categorical).
- n_components: number of principal components to compute.
- show_labels: whether to display sample labels on the PCA scatter plot.
- label_size: font size of sample labels if displayed.
- components_to_plot: tuple specifying which PCs to plot (e.g., (1,2)).
- hotelling: if True, calculates Hotelling's T² and marks multivariate outliers.
- std_scaling: whether to standardize spectral values before PCA.
- number_variable_loading: number of most influential variables to highlight in a loadings scatter plot.
- title: optional plot title.
- legendx, legendy: optional coordinates for the legend placement.
- results_path: folder to save PCA plots.
- legend_font_size: font size for the legend text.

Returns:
- pca: fitted PCA object (scikit-learn PCA) containing components, explained variance, etc.
- pca_df: DataFrame with PCA scores per sample, Q residuals, and Hotelling's T² (if requested).

Notes:
- Top influential variables are determined by the sum of squares of loadings in the selected PCs.
- Q residuals highlight samples that deviate from the PCA reconstruction.
- Hotelling's T² detects multivariate outliers in the PCA space.
- The function produces multiple plots: PCA scatter, loadings, Hotelling vs Q residuals, and explained variance.
"""


pca_result, pca_df = ef.pca_preprocessing(df_long, preprocessing_metric='Mean_SG1_W15_P2',
                                        unique_sample_variable='ID', n_components=10,
                                          show_labels=False, label_size=1,components_to_plot=(1,2), color_variable="Session", hotelling=True,
                                            std_scaling=False, number_variable_loading=200, legendx=None, legendy=None, 
                                            results_path=results_directory, legend_font_size=8)
pca_df